**Loading Secrets**

In [54]:
# Loading the environments and libraries
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


**Importing Required Libraries**

In [55]:
from openai import OpenAI
import requests
import json
from langchain_community.document_loaders import PyPDFLoader
import os
import gradio as gr

**Service 1 - Real-time Weather Assistant**

In [ ]:
# Initializing OpenAI FM
client_1 = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [57]:
# Building the back-end of the service

# API function
def get_weather(city: str):
    url = f"http://api.weatherstack.com/current?access_key=e8439ac925148b7c736a4b831729364c&query={city}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

# Define tool schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. Toronto"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

def run_model(user_prompt):
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    response = client_1.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)

        if tool_call.function.name == "get_weather":
            api_result = get_weather(args["city"])

            messages.append(message)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(api_result)
            })

            messages.append({
                "role": "system",
                "content": (
                    "Using the weather data provided, respond in exactly two sentences:\n"
                    "1. A short summary of the weather\n"
                    "2. A one-line clothing recommendation based on the weather"
                )
            })

            final_response = client_1.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages
            )

            return final_response.choices[0].message.content

    return "I'm sorry, but I’m unable to help with that request right now."


demo = gr.Interface(
    fn=run_model,
    inputs=gr.Textbox(label="Enter your prompt"),
    outputs=gr.Textbox(label="Model response"),
    title="Weather Assistant",
    description="Ask about the weather in a city and get a clothing recommendation."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


***Service 2 - Acumatica ERP Data Migration Assistant***

In [58]:
# Loading required libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters  import RecursiveCharacterTextSplitter

# Loading the corpus
file_path = "F110_Data_Migration_2025R2.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

50


In [59]:
# Chunking Process initiated below

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000, 
    chunk_overlap=200, 
    length_function = len, 
    add_start_index = True
)

chunks = text_splitter.split_documents(docs)
print(f'Split {len(docs)} pages (documents) into {len(chunks)} chunks.' )

Split 50 pages (documents) into 80 chunks.


In [60]:
chunks[0].page_content

'Consultant Course\nData Migration\nF110 Data Migration\n2025 R2\nRevision: 11/26/2025'

In [75]:
# Importing libraries required for next step
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# Converting chunks to embeddings and storing them in Chroma DB

embedding_function = OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small",
    api_base='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1'

)

client = chromadb.PersistentClient(path="f110_chroma_db")

collection = client.get_or_create_collection(
    name="f110_data_migration",
    embedding_function=embedding_function
)

documents = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]
ids = [f"id_{i}" for i in range(len(chunks))]

collection.upsert(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

results = collection.query(
    query_texts=["What are the key steps in data migration?"],
    n_results=5
)

for i in range(len(results["documents"][0])):
    print(f"\nResult {i+1}")
    print("ID:", results["ids"][0][i])
    print("Metadata:", results["metadatas"][0][i])
    print(results["documents"][0][i][:500])


PermissionDeniedError: Error code: 403 - {'message': 'Forbidden'} in upsert.